In [5]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

def find_repo_root(start=None, markers=('.git', 'environment.yml', 'requirements.txt', 'README.md')):
    """Walk upward from `start` (or cwd) until a marker file/dir is found.
    Returns absolute path to repo root or None if not found.
    """
    if start is None:
        start = os.getcwd()
    cur = os.path.abspath(start)
    while True:
        for m in markers:
            if os.path.exists(os.path.join(cur, m)):
                return cur
        parent = os.path.dirname(cur)
        if parent == cur:
            return None
        cur = parent

repo_root = find_repo_root()
if repo_root is None:
    # fallback to repository parent heuristics
    repo_root = os.path.abspath(os.path.join(os.getcwd(), '..'))

# canonical path to materials.csv inside repo: data/raw/materials.csv
DATA_PATH = os.path.join(repo_root, 'data', 'raw', 'materials.csv')
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"materials.csv not found at {DATA_PATH}. Checked repo root: {repo_root}")

print('Using data:', DATA_PATH)

Using data: /Users/nk/Downloads/GNN/xtrium-material-gnn/data/raw/materials.csv


In [ ]:

try:
    df = pd.read_csv(DATA_PATH)
    print('Loaded dataframe with', len(df), 'rows')
    display(df.head())
except Exception as e:
    raise RuntimeError(f'Failed to load {DATA_PATH}: {e}')

Loaded dataframe with 30 rows


,Id,Material_Name,Database_Source,Category,URL,Name,Database,Composition,Manufacturer,Technical_Readiness_Level,...,Replacement_Fraction_per_Flight_Value,Replacement_Fraction_per_Flight_Units,Replacement_Fraction_per_Flight_Source,Replacement_Fraction_per_Flight_STP,Replacement_Fraction_per_Flight_Last_Modified,Reuse_Flight_Limit_of_flights_Value,Reuse_Flight_Limit_of_flights_Units,Reuse_Flight_Limit_of_flights_Source,Reuse_Flight_Limit_of_flights_STP,Reuse_Flight_Limit_of_flights_Last_Modified
0,1,"1/4"" Rohacell 31 foam/carbon cloth Aircraft Sk...",NASA Langley Research Center Database,Structural Organic Composites | Description:,https://tpsx.arc.nasa.gov/Material?id=1547,"1/4"" Rohacell 31 foam/carbon cloth Aircraft Sk...",NASA Langley Research Center Database | Descri...,Description:,Description:,4 | Description:,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,"1,3-Bis(3,4-dicarboxyphenoxy-4-phenyl-2-propyl...",NASA Langley Research Center Database,Polyamides | Description:,https://tpsx.arc.nasa.gov/Material?id=1437,"1,3-Bis(3,4-dicarboxyphenoxy-4-phenyl-2-propyl...",NASA Langley Research Center Database | Descri...,"Composition of BMDEDA - 1,3-Bis(3,4-dicarboxyp...","1,3-Bis(4-hydroxyphenyl-2-propyl)benzene (bish...",Description:,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,"1,3-Bis(3,4-dicarboxyphenoxy-4-phenyl-2-propyl...",NASA Langley Research Center Database,Polyamides | Description:,https://tpsx.arc.nasa.gov/Material?id=1441,"1,3-Bis(3,4-dicarboxyphenoxy-4-phenyl-2-propyl...",NASA Langley Research Center Database | Descri...,"Composition of BMDEDA - 1,3-Bis(3,4-dicarboxyp...","1,3-Bis(4-hydroxyphenyl-2-propyl)benzene (bish...",Description:,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,"1,3-Bis(3,4-dicarboxyphenoxy-4-phenyl-2-propyl...",NASA Langley Research Center Database,Polyamides | Description:,https://tpsx.arc.nasa.gov/Material?id=1440,"1,3-Bis(3,4-dicarboxyphenoxy-4-phenyl-2-propyl...",NASA Langley Research Center Database | Descri...,"Composition of BMDEDA - 1,3-Bis(3,4-dicarboxyp...","1,3-Bis(4-hydroxyphenyl-2-propyl)benzene (bish...",Description:,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,"1,3-Bis(3,4-dicarboxyphenoxy-4-phenyl-2-propyl...",NASA Langley Research Center Database,Polyamides | Description:,https://tpsx.arc.nasa.gov/Material?id=1438,"1,3-Bis(3,4-dicarboxyphenoxy-4-phenyl-2-propyl...",NASA Langley Research Center Database | Descri...,"Composition of BMDEDA - 1,3-Bis(3,4-dicarboxyp...","1,3-Bis(4-hydroxyphenyl-2-propyl)benzene (bish...",Description:,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
df.shape

(30, 255)

In [8]:

display(df.head(10))
display(df.dtypes)

,Id,Material_Name,Database_Source,Category,URL,Name,Database,Composition,Manufacturer,Technical_Readiness_Level,...,Replacement_Fraction_per_Flight_Value,Replacement_Fraction_per_Flight_Units,Replacement_Fraction_per_Flight_Source,Replacement_Fraction_per_Flight_STP,Replacement_Fraction_per_Flight_Last_Modified,Reuse_Flight_Limit_of_flights_Value,Reuse_Flight_Limit_of_flights_Units,Reuse_Flight_Limit_of_flights_Source,Reuse_Flight_Limit_of_flights_STP,Reuse_Flight_Limit_of_flights_Last_Modified
0,1,"1/4"" Rohacell 31 foam/carbon cloth Aircraft Sk...",NASA Langley Research Center Database,Structural Organic Composites | Description:,https://tpsx.arc.nasa.gov/Material?id=1547,"1/4"" Rohacell 31 foam/carbon cloth Aircraft Sk...",NASA Langley Research Center Database | Descri...,Description:,Description:,4 | Description:,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,"1,3-Bis(3,4-dicarboxyphenoxy-4-phenyl-2-propyl...",NASA Langley Research Center Database,Polyamides | Description:,https://tpsx.arc.nasa.gov/Material?id=1437,"1,3-Bis(3,4-dicarboxyphenoxy-4-phenyl-2-propyl...",NASA Langley Research Center Database | Descri...,"Composition of BMDEDA - 1,3-Bis(3,4-dicarboxyp...","1,3-Bis(4-hydroxyphenyl-2-propyl)benzene (bish...",Description:,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,"1,3-Bis(3,4-dicarboxyphenoxy-4-phenyl-2-propyl...",NASA Langley Research Center Database,Polyamides | Description:,https://tpsx.arc.nasa.gov/Material?id=1441,"1,3-Bis(3,4-dicarboxyphenoxy-4-phenyl-2-propyl...",NASA Langley Research Center Database | Descri...,"Composition of BMDEDA - 1,3-Bis(3,4-dicarboxyp...","1,3-Bis(4-hydroxyphenyl-2-propyl)benzene (bish...",Description:,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,"1,3-Bis(3,4-dicarboxyphenoxy-4-phenyl-2-propyl...",NASA Langley Research Center Database,Polyamides | Description:,https://tpsx.arc.nasa.gov/Material?id=1440,"1,3-Bis(3,4-dicarboxyphenoxy-4-phenyl-2-propyl...",NASA Langley Research Center Database | Descri...,"Composition of BMDEDA - 1,3-Bis(3,4-dicarboxyp...","1,3-Bis(4-hydroxyphenyl-2-propyl)benzene (bish...",Description:,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,"1,3-Bis(3,4-dicarboxyphenoxy-4-phenyl-2-propyl...",NASA Langley Research Center Database,Polyamides | Description:,https://tpsx.arc.nasa.gov/Material?id=1438,"1,3-Bis(3,4-dicarboxyphenoxy-4-phenyl-2-propyl...",NASA Langley Research Center Database | Descri...,"Composition of BMDEDA - 1,3-Bis(3,4-dicarboxyp...","1,3-Bis(4-hydroxyphenyl-2-propyl)benzene (bish...",Description:,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,6,"1,3-Bis(3,4-dicarboxyphenoxy-4-phenyl-2-propyl...",NASA Langley Research Center Database,Polyamides | Description:,https://tpsx.arc.nasa.gov/Material?id=1439,"1,3-Bis(3,4-dicarboxyphenoxy-4-phenyl-2-propyl...",NASA Langley Research Center Database | Descri...,"Composition of BMDEDA - 1,3-Bis(3,4-dicarboxyp...","1,3-Bis(4-hydroxyphenyl-2-propyl)benzene (bish...",Description:,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,7,"3,3',4,4'-Benzophenone tetracarboxylic dianhyd...",NASA Langley Research Center Database,Polyamides | Description:,https://tpsx.arc.nasa.gov/Material?id=1499,"3,3',4,4'-Benzophenone tetracarboxylic dianhyd...",NASA Langley Research Center Database | Descri...,Description:,"4,4'-Oxydiphthalic anhydride (ODPA) and Hydro...",Description:,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,8,35v/o SCS-9a-(20v/oRBSiC/ZrB2),NASA Ames Thermal Protection Materials,Ultra-High Temperature Ceramics | Description:,https://tpsx.arc.nasa.gov/Material?id=31,35v/o SCS-9a-(20v/oRBSiC/ZrB2) | Description:,NASA Ames Thermal Protection Materials | Descr...,Description:,"Advanced Ceramics Research, Tucson, Arizona. |...",Description:,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,9,"4,4'-(2-diphenylphosphinyl-1,4-phenylenedioxy)...",NASA Langley Research Center Database,Polyamides | Description:,https://tpsx.arc.nasa.gov/Material?id=1494,"4,4'-(2-diphenylphosphinyl-1,4-phenylenedioxy)...",NASA Langley Res

Id                                               int64
Material_Name                                   object
Database_Source                                 object
Category                                        object
URL                                             object
                                                ...   
Reuse_Flight_Limit_of_flights_Value            float64
Reuse_Flight_Limit_of_flights_Units             object
Reuse_Flight_Limit_of_flights_Source            object
Reuse_Flight_Limit_of_flights_STP               object
Reuse_Flight_Limit_of_flights_Last_Modified     object
Length: 255, dtype: object

In [9]:
missing = df.isnull().mean().sort_values(ascending=False)
print("Top missing cols:\n", missing.head(20))
numeric = df.select_dtypes(include=['float','int'])
display(numeric.describe().T)

Top missing cols:
 Tensile_Modulus_Thru_the_Thickness_10_Value            0.966667
Compressive_Strength_In_Plane_STP                      0.966667
Tensile_Modulus_Thru_the_Thickness_10_STP              0.966667
Tensile_Modulus_Thru_the_Thickness_10_Last_Modified    0.966667
Tensile_Modulus_In_Plane_11_Value                      0.966667
Tensile_Modulus_In_Plane_11_Units                      0.966667
Tensile_Modulus_In_Plane_11_Source                     0.966667
Tensile_Modulus_In_Plane_11_STP                        0.966667
Tensile_Modulus_In_Plane_11_Last_Modified              0.966667
Compressive_Strength_In_Plane_Value                    0.966667
Compressive_Strength_In_Plane_Units                    0.966667
Compressive_Strength_In_Plane_Source                   0.966667
Compressive_Strength_In_Plane_Reference                0.966667
Poisson_s_Ratio_Isotropic_13_Last_Modified             0.966667
Compressive_Strength_In_Plane_Last_Modified            0.966667
Shear_Strength_Isotro

,count,mean,std,min,25%,50%,75%,max
Id,30.0,15.500000,8.803408,1.0000,8.2500,15.5000,22.7500,30.0000
Density_Value,15.0,1367.533333,691.259603,288.0000,1120.0000,1240.0000,1450.0000,3600.0000
Density_Uncertainty,1.0,1.440000,NaN,1.4400,1.4400,1.4400,1.4400,1.4400
Density_Reference,8.0,1.000000,0.000000,1.0000,1.0000,1.0000,1.0000,1.0000
Thermal_Conductivity_Thru_the_Thickness_Value,1.0,0.107000,NaN,0.1070,0.1070,0.1070,0.1070,0.1070
...,...,...,...,...,...,...,...,...
Purchase_Cost_Value,1.0,129000.000000,NaN,129000.0000,129000.0000,129000.0000,129000.0000,129000.0000
Installation_Time_Value,1.0,372000.000000,NaN,372000.0000,372000.0000,372000.0000,372000.0000,372000.0000
Inspection_Repair_Time_per_Flight_Value,1.0,4260.000000,NaN,4260.0000,4260.0000,4260.0000,4260.0000,4260.0000
Replacement_Fraction_per_Flight_Value,1.0,0.001300,NaN,0.0013,0.0013,0.0013,0.0013,0.0013


In [10]:
text_cols = [c for c in df.columns if df[c].dtype=="object" and df[c].nunique()>20]
cat_cols  = [c for c in df.columns if df[c].dtype=="object" and df[c].nunique()<=20]
num_cols  = numeric.columns.tolist()
print("Text cols:", text_cols)
print("Categorical cols:", cat_cols)
print("Numeric cols:", num_cols)

Text cols: ['Material_Name', 'URL', 'Name', 'Description']
Categorical cols: ['Database_Source', 'Category', 'Database', 'Composition', 'Manufacturer', 'Technical_Readiness_Level', 'Last_Modified', 'Point_of_Contact', 'Property_References', 'General_References', 'Density_Units', 'Density_Source', 'Density_STP', 'Density_Last_Modified', 'Thermal_Conductivity_Thru_the_Thickness_Units', 'Thermal_Conductivity_Thru_the_Thickness_Source', 'Thermal_Conductivity_Thru_the_Thickness_STP', 'Thermal_Conductivity_Thru_the_Thickness_Last_Modified', 'Thermal_Conductivity_In_Plane_Units', 'Thermal_Conductivity_In_Plane_Source', 'Thermal_Conductivity_In_Plane_STP', 'Thermal_Conductivity_In_Plane_Last_Modified', 'Specific_Heat_Units', 'Specific_Heat_Source', 'Specific_Heat_STP', 'Specific_Heat_Last_Modified', 'Notes', 'Tensile_Strength_Isotropic_1_Units', 'Tensile_Strength_Isotropic_1_Source', 'Tensile_Strength_Isotropic_1_STP', 'Tensile_Strength_Isotropic_1_Last_Modified', 'Tensile_Yield_Strength_Isotr

In [11]:
print("Column sample names:", df.columns.tolist())
prop_candidates = [c for c in df.columns if any(unit in c.lower() for unit in ['mpa','pa','kg','g','m3','%','wt'])]
print("Prop name candidates:", prop_candidates)

Column sample names: ['Id', 'Material_Name', 'Database_Source', 'Category', 'URL', 'Name', 'Database', 'Composition', 'Manufacturer', 'Technical_Readiness_Level', 'Last_Modified', 'Description', 'Point_of_Contact', 'Property_References', 'General_References', 'Density_Value', 'Density_Units', 'Density_Uncertainty', 'Density_Source', 'Density_STP', 'Density_Reference', 'Density_Last_Modified', 'Thermal_Conductivity_Thru_the_Thickness_Value', 'Thermal_Conductivity_Thru_the_Thickness_Units', 'Thermal_Conductivity_Thru_the_Thickness_Uncertainty', 'Thermal_Conductivity_Thru_the_Thickness_Source', 'Thermal_Conductivity_Thru_the_Thickness_STP', 'Thermal_Conductivity_Thru_the_Thickness_Reference', 'Thermal_Conductivity_Thru_the_Thickness_Last_Modified', 'Thermal_Conductivity_In_Plane_Value', 'Thermal_Conductivity_In_Plane_Units', 'Thermal_Conductivity_In_Plane_Uncertainty', 'Thermal_Conductivity_In_Plane_Source', 'Thermal_Conductivity_In_Plane_STP', 'Thermal_Conductivity_In_Plane_Reference', '